# 04 -- Collaborative Filtering (implicit ALS)

Matrix-factorization collaborative filtering via the **`implicit`** library
(`implicit.als.AlternatingLeastSquares`), evaluated with **precision@10**.

### Algorithm

Implicit-feedback ALS (Hu, Koren & Volinsky 2008):
1. Build a sparse user x song matrix from play counts; each cell is a
   confidence `c_ui = alpha * play_count`.
2. `AlternatingLeastSquares` alternates least-squares updates to extract
   latent user and song factors that best explain the play signal.
3. **4a -- user-based**: score every song for a user as `U_u . V_i`, recommend
   the top-10 the user hasn't played.
4. **4b -- item-based**: for a seed song, rank songs by cosine similarity of
   their latent profiles, recommend the top-10.

### Evaluation
- **Train/test split**: random holdout of 20% per user (no timestamps in
  the data, so "last 20%" is approximated by a random split).
- **Metric**: Precision@10 -- fraction of recommended tracks the user
  actually listened to in the test set. Target: **> 10%**.

### Output columns
| Column | Description |
|--------|-------------|
| `rank` | 1-10, by likelihood score (descending) |
| `artist` | Artist name |
| `title` | Track title |

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

from src.data.loader import MySpotifyRecommender

from src.models.collaborative_filtering import (
    build_user_item_matrix,
    fit_als,
    evaluate_user_cf,
)
from implicit.evaluation import train_test_split


/home/samy/MySpotify/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/samy/MySpotify/.venv/lib/python3.12/site-packages/implicit/gpu/__init__.py:28: UserWarning: Disabling GPU support because of 'libcublas.so.13: cannot open shared object file: No such file or directory'
  warnings.warn(


In [2]:
rs = MySpotifyRecommender.from_files(
    data_dir=Path.cwd().parent / "data",
    download=True,
    # triplets_sample_rows=1_000_000
)

Data dir    : /home/samy/MySpotify/data
Source      : cleaned CSVs (/home/samy/MySpotify/data/csv)



  tracks      (1000000, 4)
  genres      (280831, 3)
  triplets    (48373586, 3)
  lyrics_long (16845822, 3)


---
## Research

### Train / Test Split

In [3]:
user_item, user_idx, song_idx, idx_song = build_user_item_matrix(rs.triplets)
user_item.shape

(1019318, 384546)

In [4]:
train, test = train_test_split(user_item, train_percentage=0.8, random_state=42)

### 4. Collaborative Filtering

In [5]:
model = fit_als(train, factors=192, regularization=0.09, alpha=1.0, iterations=25)

/home/samy/MySpotify/.venv/lib/python3.12/site-packages/implicit/cpu/als.py:96: RuntimeWarning: OpenBLAS is configured to use 28 threads. It is highly recommended to disable its internal threadpool by setting the environment variable 'OPENBLAS_NUM_THREADS=1' or by calling 'threadpoolctl.threadpool_limits(1, "blas")'. Having OpenBLAS use a threadpool can lead to severe performance issues here.
  check_blas_config()


  0%|          | 0/25 [00:00<?, ?it/s]

  4%|▍         | 1/25 [00:06<02:25,  6.06s/it]

  8%|▊         | 2/25 [00:12<02:23,  6.22s/it]

 12%|█▏        | 3/25 [00:18<02:20,  6.37s/it]

...

 88%|████████▊ | 22/25 [02:45<00:26,  8.79s/it]

 92%|█████████▏| 23/25 [02:55<00:18,  9.07s/it]

 96%|█████████▌| 24/25 [03:05<00:09,  9.23s/it]

100%|██████████| 25/25 [03:14<00:00,  9.30s/it]

100%|██████████| 25/25 [03:14<00:00,  7.79s/it]

### 4a -- User-based recommendations (latent factors)

In [6]:
sample_user = ''
for i in user_idx.keys():
    sample_user = i
    break
print(f"Sample user: {sample_user}")

uid = user_idx[sample_user]

top_pred_indices, _ = model.recommend(
    uid, 
    train[uid],
    N=10,
    filter_already_liked_items=True
)

top_pred_item_ids = [idx_song[idx] for idx in top_pred_indices]

df = rs.tracks[rs.tracks["song_id"].isin(top_pred_item_ids)].reset_index(drop=True)
df = df[["artist", "title", "song_id"]]
df = df.drop_duplicates(subset=["song_id"])
df[["artist", "title"]]

Sample user: 00000b722001882066dff9d2da8a775658053ea0


,artist,title
0,Edwyn Collins,You'll Never Know (My Love) (Bovellian 07 Mix)
1,Edwyn Collins,Superstar Talking Blues
2,The Buggles,Video Killed The Radio Star
3,Atomic Kitten,Eternal Flame (Single Version)
4,Fisher,Rianna
5,Valerio Scanu,Esisti Tu
6,Barry Tuckwell/Academy of St Martin-in-the-Fie...,Horn Concerto No. 4 in E flat K495: II. Romanc...
7,Cosmo Vitelli,Robot Soul (Radio Edit)
8,Suicidal Tendencies,Go Skate! (Possessed To Skate '97)
9,The Verve,Lord I Guess I'll Never Know


### 4b -- Similar tracks (item-based CF)

In [7]:
top_song = top_pred_item_ids[1]

print(f"Top recommended song for user {sample_user}: {top_song}")

item_id = song_idx[top_song]

print(f"\n{rs.tracks[rs.tracks['song_id'] == top_song][['artist', 'title']].iloc[0]}")

similar_indices, similarity_scores = model.similar_items(itemid=item_id, N=11)
similar_indices = similar_indices[1:]
similarity_scores = similarity_scores[1:]

similar_item_ids = [idx_song[idx] for idx in similar_indices]

df = rs.tracks[rs.tracks["song_id"].isin(similar_item_ids)].reset_index(drop=True)
df = df.drop_duplicates(subset=["song_id"])
df[["artist", "title"]]

Top recommended song for user 00000b722001882066dff9d2da8a775658053ea0: SONHWUN12AC468C014

artist    Fisher
title     Rianna
Name: 456530, dtype: str


,artist,title
0,N.E.R.D.,Rock Star
1,Chris Isaak,The Christmas Song
2,Mickie Krause,Orange Trägt Nur Die Müllabfuhr (Go West)
3,Sonny Boy Williamson,Don't Start Me Talkin'
4,Switchblade Symphony,Dollhouse
5,Cartola,Tive Sim
6,Charlelie Couture,L'histoire De Bernard Workers
7,Taylor Swift,You Belong With Me
8,Taylor Swift,Love Story
9,Brenda Lee,Rockin' Around The Christmas Tree


#### Precision@10 -- User-based & Item-Based CF

In [8]:
pk_eval = evaluate_user_cf(
    model,
    train,
    test,
)

pk_eval

  0%|          | 0/998833 [00:00<?, ?it/s]

  0%|          | 1000/998833 [00:01<25:18, 657.29it/s]

  0%|          | 2000/998833 [00:03<25:19, 655.82it/s]

  0%|          | 3000/998833 [00:04<27:06, 612.16it/s]

  0%|          | 4000/998833 [00:06<25:18, 655.14it/s]

...

100%|█████████▉| 997000/998833 [23:54<00:02, 655.11it/s]

100%|█████████▉| 998000/998833 [23:56<00:01, 647.11it/s]

100%|██████████| 998833/998833 [23:57<00:00, 639.67it/s]

100%|██████████| 998833/998833 [23:57<00:00, 694.76it/s]

0.11717241827367085